In [ ]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5fbd0217852022e72d5569b16e0f32dfb3f9b9fc20dd98824738ab9c41995c95
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [ ]:
import torch
import pandas as pd
import numpy as np
import evaluate
import nltk
import gc
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    PegasusTokenizer,
    PegasusForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

from google.colab import drive, files
drive.mount('/content/drive')

nltk.download("punkt")
nltk.download("punkt_tab")

Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Device

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


Load Dataset

In [ ]:
print("Loading pre-mixed dataset...")
df_mixed = pd.read_csv("/content/drive/MyDrive/University/Natural Language Processing/NLP Project/MixedDataset/mixed_meeting_dataset.csv")

print(f"Total Mixed Dataset Size: {len(df_mixed)}")
print(df_mixed.head())

full_dataset = Dataset.from_pandas(df_mixed)


# Split into Train and Validation (80% Train, 20% Eval)
split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

Loading pre-mixed dataset...


Tokenizer

In [ ]:
model_checkpoint = "google/pegasus-xsum"

tokenizer = PegasusTokenizer.from_pretrained(model_checkpoint)

model = PegasusForConditionalGeneration.from_pretrained(
    model_checkpoint
).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

Preprocessing

In [ ]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["Transcript"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=examples["Summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing training dataset...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)

print("Tokenizing eval dataset...")
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

Tokenizing training dataset...


Map:   0%|          | 0/1168 [00:00<?, ? examples/s]

Tokenizing eval dataset...


Map:   0%|          | 0/292 [00:00<?, ? examples/s]

Data Collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Rouge Metrics

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = predictions.astype(np.int64)
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = ["\n".join(nltk.sent_tokenize(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(l.strip())) for l in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result["gen_len"] = np.mean([np.count_nonzero(p != tokenizer.pad_token_id) for p in predictions])

    return {k: round(v, 4) for k, v in result.items()}

Training Arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/pegasus_output",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=1,
    fp16=True,
    logging_steps=100,
    report_to="none"
)

Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Training

In [ ]:
trainer.train()

Evaluation

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
eval_results = trainer.evaluate()

print("\n=== FINAL RESULTS ===")

for key, value in eval_results.items():
    print(f"{key}: {value}")

Training Loss,Validation Loss,Step,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
7.937252,1.835351,876,0.460400,0.280300,0.403700,0.423400,35.113000



=== FINAL RESULTS ===
eval_loss: 1.8353513479232788
eval_rouge1: 0.4604
eval_rouge2: 0.2803
eval_rougeL: 0.4037
eval_rougeLsum: 0.4234
eval_gen_len: 35.113


Testing

In [ ]:
# Read transcript file
with open(
    "/content/drive/MyDrive/University/Natural Language Processing/NLP Project/MixedDataset/Test3_Project_Progress.txt",
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

print("INPUT LENGTH:", len(text))
print("\nFIRST 500 CHARS:\n")
print(text[:500])

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
).to(DEVICE)

print("\nTOKENIZED SHAPE:")
print(inputs["input_ids"].shape)

# Generate
with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4,
        early_stopping=True
    )

print("\nRAW TOKEN IDS:")
print(summary_ids)

# Decode
summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("\nGENERATED SUMMARY:")
print(repr(summary))

[transformers] Both `max_new_tokens` (=64) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INPUT LENGTH: 1960

FIRST 500 CHARS:

Speaker 1: Good morning. Let's begin today's project meeting. The main objective is to review our progress on the meeting summarization system.

Speaker 2: Good morning. I have completed the data preprocessing stage. The transcripts were cleaned, speaker labels were standardized, and the dataset was split into training, validation, and testing sets.

Speaker 1: That's good progress. How is the model training going?

Speaker 2: We tested several transformer-based models including BART, PEGASUS, a

TOKENIZED SHAPE:
torch.Size([1, 256])

RAW TOKEN IDS:
tensor([[   0,  139,  674, 4129,  117,  112,  933,  150, 1974,  124,  109,  988,
         5906, 6520, 3884,  327,  107,  139,  674, 4129,  117,  112,  933,  150,
         1974,  124,  109,  988, 5906, 6520, 3884,  327,  107,    1]],
       device='cuda:0')

GENERATED SUMMARY:
'The main objective is to review our progress on the meeting summarization system. The main objective is to review our progress o

Save Model

In [ ]:
SAVE_PATH = "/content/pegasus_final_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"\nModel saved to: {SAVE_PATH}")

print("\nZipping model...")
!zip -r /content/pegasus_final_model.zip /content/pegasus_final_model

print("\nDownloading model to computer...")

files.download("/content/pegasus_final_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: /content/pegasus_final_model

Zipping model...
  adding: content/pegasus_final_model/ (stored 0%)
  adding: content/pegasus_final_model/model.safetensors (deflated 7%)
  adding: content/pegasus_final_model/generation_config.json (deflated 58%)
  adding: content/pegasus_final_model/config.json (deflated 62%)
  adding: content/pegasus_final_model/tokenizer_config.json (deflated 78%)
  adding: content/pegasus_final_model/tokenizer.json (deflated 78%)
  adding: content/pegasus_final_model/training_args.bin (deflated 53%)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>